## Solution for Assignment 02 Question 3

### Setup

In [12]:
import torch
import torch.nn as nn
from torchvision.models import resnet18
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

### CONSTANTS ###
BATCH_SIZE = 256
EPOCHS = 10
LR = 1e-3

### MODEL ###
# Initialize the ResNet18 model with random weights using torchvision
resnet18_model = resnet18(weights=None)
# Adjust input and output layers to work with FMNIST
resnet18_model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
resnet18_model.fc = nn.Linear(resnet18_model.fc.in_features, 10)

In [13]:
### DATA ###

# Transform to convert FMNIST data into normalized tensors
fmnist_data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Init train and test datasets
train_data = datasets.FashionMNIST(
    root='/datasets/FMNIST/',
    train=True,
    download=True,
    transform=fmnist_data_transform
)

test_data = datasets.FashionMNIST(
    root='/datasets/FMNIST/',
    train=False,
    download=True,
    transform=fmnist_data_transform
)

# Define dataloaders
FMNIST_train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
FMNIST_test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

### a)

In [ ]:
### TRAIN and TEST ###
LAMBDA = 0.0001

def train_l2_regularize(epochs, learning_rate, model, train_loader, test_loader, lambda_val):
    test_accs = []
    # Define the optimizer
    optim = torch.optim.Adam(
        params=model.parameters(),
        lr=learning_rate,
        weight_decay=0
    )
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()

        for features, labels in train_loader:
            # Calculate original cross entropy loss
            output_model = model(features)
            loss = criterion(output_model, labels)
            # Add regularization term
            loss += lambda_val * sum((p**2).sum() for p in model.parameters()) / 2

            # Backpropagate
            optim.zero_grad()
            loss.backward()
            optim.step()

        model.eval()

        total = 0
        correct = 0
        for features, labels in test_loader:
            output_model_test = model(features)
            predictions = output_model_test.argmax(axis=1)
            total += output_model_test.size(0)
            correct += (predictions == labels).sum().item()

        test_acc = correct / total
        print(f"Epoch {epoch +1}: Test accuracy = {test_acc:.4f}")
        test_accs.append(test_acc)
    return test_accs


for lambda_val in (0.01, 0.001, 0.0001):
    for lr in (0.01, 0.001, 0.0001):
        test_accs = train_l2_regularize(
            EPOCHS, lr, resnet18_model, FMNIST_train_loader, FMNIST_test_loader, lambda_val
        )
        last_test_acc = test_accs[-1]
        aver_last_10_test_accs = sum(test_accs[-10:]) / len(test_accs[-10:])
        aver_test_acc = sum(test_accs) / len(test_accs)
        print(f'For lambda: {lambda_val}, lr: {lr}, final acc: {last_test_acc}, aver last 10: {aver_last_10_test_accs}, aver test acc: {aver_test_acc}')
    
        

KeyboardInterrupt: 